
# FairWarn-SHS — Notebook 12
## Graph-aware SHAP Explainability

### Purpose

This notebook explains the final **FairWarn-SHS GraphSAGE model**.

The interpretation is deliberately narrow and defensible:

> SHAP values estimate how the **target student's own encoded features** affect
> that student's predicted at-risk probability while the graph structure and
> all other students' features are held fixed.

This means the explanation is **conditional on the student's graph neighbourhood**.

It does **not** claim that SHAP directly explains which graph edges are causal.

### Outputs

The notebook creates:

1. Global SHAP feature importance.
2. Local SHAP explanations for available:
   - true positive;
   - false positive;
   - true negative;
   - false negative.
3. A metadata file recording the representative seed and explanation design.
4. Publication-quality PNG figures at 300 dpi.

### Model

The FairWarn-SHS loss remains:

\[
L_{\mathrm{FairWarn}}
=
L_{\mathrm{weighted\ CE}}
+
2.0\,L_{\mathrm{EO}}
\]

where the residence equal-opportunity regularizer is the single fairness-aware
model modification.


In [ ]:
!pip -q install torch-geometric shap pandas numpy scikit-learn matplotlib

In [ ]:

from google.colab import files

uploaded = files.upload()

# Upload these three files:
# 1. FairWarn_SHS_Node_Features.csv
# 2. FairWarn_SHS_Edge_List.csv
# 3. fairwarn11_harmonized_metrics_by_seed.csv


In [ ]:

import json
import random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import shap

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

NODE_FILE = "FairWarn_SHS_Node_Features.csv"
EDGE_FILE = "FairWarn_SHS_Edge_List.csv"
METRICS_FILE = "fairwarn11_harmonized_metrics_by_seed.csv"

FAIRNESS_LAMBDA = 2.0
MAX_EPOCHS = 500
PATIENCE = 40

OUTPUT_DIR = Path("outputs/explainability")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



## Step 1 — Choose a representative FairWarn-SHS seed

Instead of choosing the best seed, which would make the explanation look
artificially favourable, the notebook selects the FairWarn-SHS run whose
AUC-PR is **closest to the five-seed mean AUC-PR** from Notebook 11.

This provides a representative run rather than a cherry-picked one.


In [ ]:

metrics11 = pd.read_csv(METRICS_FILE)

fairwarn_runs = (
    metrics11[
        metrics11["Model"].eq("FairWarn-SHS")
    ]
    .copy()
    .sort_values("Seed")
)

fairwarn_mean_ap = fairwarn_runs["AUC_PR"].mean()

fairwarn_runs["Distance_From_Mean_AUC_PR"] = (
    fairwarn_runs["AUC_PR"] - fairwarn_mean_ap
).abs()

representative_row = fairwarn_runs.sort_values(
    ["Distance_From_Mean_AUC_PR", "Seed"]
).iloc[0]

REPRESENTATIVE_SEED = int(representative_row["Seed"])

print("Five-seed FairWarn mean AUC-PR:", round(fairwarn_mean_ap, 4))
print("Representative seed:", REPRESENTATIVE_SEED)
print("Representative AUC-PR:", round(float(representative_row["AUC_PR"]), 4))


## Step 2 — Reconstruct the same Ghana graph and common split

In [ ]:

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

labelled_mask = (
    nodes["Label_Available"].eq(1)
    & nodes["TARGET_AtRisk"].notna()
).to_numpy()

excluded = {
    "Node_ID",
    "Roster_Code",
    "School_Code",
    "Class_Code",
    "Label_Available",
    "TARGET_AtRisk",
}

feature_columns = [
    c for c in nodes.columns
    if c not in excluded
]

y_all = (
    nodes["TARGET_AtRisk"]
    .fillna(-1)
    .astype(int)
    .to_numpy()
)

residence_text = (
    nodes["Q6_Residence"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .str.lower()
)

residence = np.full(
    len(nodes),
    -1,
    dtype=np.int64,
)

residence[
    residence_text.str.contains(
        "board",
        regex=False,
    )
] = 0

residence[
    residence_text.str.contains(
        "day",
        regex=False,
    )
] = 1

node_map = {
    node_id: index
    for index, node_id
    in enumerate(nodes["Node_ID"])
}

pairs = []

for _, row in edges.iterrows():
    source_id = row["Source_Node_ID"]
    target_id = row["Target_Node_ID"]

    if (
        source_id in node_map
        and target_id in node_map
    ):
        source = node_map[source_id]
        target = node_map[target_id]

        pairs.extend([
            (source, target),
            (target, source),
        ])

edge_index = torch.tensor(
    pairs,
    dtype=torch.long,
).t().contiguous()

print("Nodes:", len(nodes))
print("Labelled:", int(labelled_mask.sum()))
print("Undirected edges:", edge_index.shape[1] // 2)


In [ ]:

def common_split(seed):
    labelled_indices = np.where(
        labelled_mask
    )[0]

    labelled_y = y_all[
        labelled_indices
    ]

    train_val, test = train_test_split(
        labelled_indices,
        test_size=0.20,
        stratify=labelled_y,
        random_state=seed,
    )

    train_val_y = y_all[
        train_val
    ]

    train, validation = train_test_split(
        train_val,
        test_size=0.1875,
        stratify=train_val_y,
        random_state=seed,
    )

    return train, validation, test


def make_preprocessor(frame):
    numeric = frame.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical = [
        c for c in frame.columns
        if c not in numeric
    ]

    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]),
            numeric,
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
    ])


## Step 3 — Train the representative FairWarn-SHS model

In [ ]:

class FairWarnGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            64,
            aggr="mean",
        )

        self.conv2 = SAGEConv(
            64,
            32,
            aggr="mean",
        )

        self.classifier = torch.nn.Linear(
            32,
            2,
        )

        self.dropout = 0.35

    def forward(
        self,
        x,
        edge_index,
    ):
        x = F.relu(
            self.conv1(
                x,
                edge_index,
            )
        )

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        x = F.relu(
            self.conv2(
                x,
                edge_index,
            )
        )

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        return self.classifier(x)


def soft_eo_loss(
    probability,
    labels,
    residence_tensor,
    train_mask,
):
    positive = (
        train_mask
        & labels.eq(1)
        & residence_tensor.ge(0)
    )

    boarding = (
        positive
        & residence_tensor.eq(0)
    )

    day = (
        positive
        & residence_tensor.eq(1)
    )

    if (
        boarding.sum() == 0
        or day.sum() == 0
    ):
        return torch.tensor(
            0.0,
            device=probability.device,
        )

    return torch.abs(
        probability[boarding].mean()
        - probability[day].mean()
    )


In [ ]:

set_seed(REPRESENTATIVE_SEED)

train_idx, validation_idx, test_idx = common_split(
    REPRESENTATIVE_SEED
)

X_raw_all = nodes[
    feature_columns
].copy()

preprocessor = make_preprocessor(
    X_raw_all.iloc[train_idx]
)

preprocessor.fit(
    X_raw_all.iloc[train_idx]
)

X_all_encoded = preprocessor.transform(
    X_raw_all
).astype(np.float32)

encoded_feature_names = (
    preprocessor
    .get_feature_names_out()
    .tolist()
)

graph = Data(
    x=torch.tensor(
        X_all_encoded,
        dtype=torch.float32,
    ),
    edge_index=edge_index,
    y=torch.tensor(
        y_all,
        dtype=torch.long,
    ),
    residence=torch.tensor(
        residence,
        dtype=torch.long,
    ),
)

train_mask = torch.zeros(
    len(nodes),
    dtype=torch.bool,
)
validation_mask = torch.zeros(
    len(nodes),
    dtype=torch.bool,
)
test_mask = torch.zeros(
    len(nodes),
    dtype=torch.bool,
)

train_mask[train_idx] = True
validation_mask[validation_idx] = True
test_mask[test_idx] = True

graph.train_mask = train_mask
graph.validation_mask = validation_mask
graph.test_mask = test_mask

graph = graph.to(device)

train_labels = graph.y[
    graph.train_mask
]

counts = torch.bincount(
    train_labels,
    minlength=2,
).float()

class_weights = (
    counts.sum()
    / (
        2.0
        * counts.clamp_min(1.0)
    )
).to(device)

model = FairWarnGraphSAGE(
    graph.num_node_features
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=5e-4,
)

best_state = None
best_validation_ap = -np.inf
best_epoch = 0
wait = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    logits = model(
        graph.x,
        graph.edge_index,
    )

    probability = torch.softmax(
        logits,
        dim=1,
    )[:, 1]

    ce_loss = F.cross_entropy(
        logits[graph.train_mask],
        graph.y[graph.train_mask],
        weight=class_weights,
    )

    fairness_loss = soft_eo_loss(
        probability,
        graph.y,
        graph.residence,
        graph.train_mask,
    )

    loss = (
        ce_loss
        + FAIRNESS_LAMBDA
        * fairness_loss
    )

    loss.backward()
    optimizer.step()

    model.eval()

    with torch.no_grad():
        logits = model(
            graph.x,
            graph.edge_index,
        )

        probability = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        validation_true = (
            graph.y[
                graph.validation_mask
            ]
            .cpu()
            .numpy()
        )

        validation_probability = (
            probability[
                graph.validation_mask
            ]
            .cpu()
            .numpy()
        )

        validation_ap = (
            average_precision_score(
                validation_true,
                validation_probability,
            )
        )

    if (
        validation_ap
        > best_validation_ap + 1e-6
    ):
        best_validation_ap = (
            validation_ap
        )
        best_epoch = epoch
        best_state = deepcopy(
            model.state_dict()
        )
        wait = 0
    else:
        wait += 1

    if wait >= PATIENCE:
        break

model.load_state_dict(
    best_state
)

model.eval()

with torch.no_grad():
    final_logits = model(
        graph.x,
        graph.edge_index,
    )

    final_probability = torch.softmax(
        final_logits,
        dim=1,
    )[:, 1]

    final_prediction = torch.argmax(
        final_logits,
        dim=1,
    )

print("Best epoch:", best_epoch)
print(
    "Representative test AUC-PR:",
    round(
        average_precision_score(
            graph.y[graph.test_mask].cpu().numpy(),
            final_probability[graph.test_mask].cpu().numpy(),
        ),
        4,
    ),
)



## Step 4 — Select local explanation cases

The notebook looks for one available example of each:

- True Positive (TP)
- False Positive (FP)
- True Negative (TN)
- False Negative (FN)

No participant names or identifying roster values are placed on the figures.
Only anonymised node indices are used.


In [ ]:

test_truth = (
    graph.y[
        graph.test_mask
    ]
    .cpu()
    .numpy()
)

test_probability = (
    final_probability[
        graph.test_mask
    ]
    .cpu()
    .numpy()
)

test_prediction = (
    final_prediction[
        graph.test_mask
    ]
    .cpu()
    .numpy()
)

case_table = pd.DataFrame({
    "Node_Index": test_idx,
    "True_Label": test_truth,
    "Predicted_Label": test_prediction,
    "AtRisk_Probability": test_probability,
})

def case_type(row):
    if (
        row["True_Label"] == 1
        and row["Predicted_Label"] == 1
    ):
        return "True Positive"

    if (
        row["True_Label"] == 0
        and row["Predicted_Label"] == 1
    ):
        return "False Positive"

    if (
        row["True_Label"] == 0
        and row["Predicted_Label"] == 0
    ):
        return "True Negative"

    return "False Negative"

case_table["Case_Type"] = case_table.apply(
    case_type,
    axis=1,
)

selected_cases = []

for desired_type in [
    "True Positive",
    "False Positive",
    "True Negative",
    "False Negative",
]:
    candidates = case_table[
        case_table["Case_Type"].eq(
            desired_type
        )
    ].copy()

    if candidates.empty:
        print(
            desired_type,
            "not available in this test split."
        )
        continue

    # Prefer a reasonably confident example,
    # but do not select using protected attributes.
    if desired_type in [
        "True Positive",
        "False Positive",
    ]:
        chosen = candidates.sort_values(
            "AtRisk_Probability",
            ascending=False,
        ).iloc[0]
    else:
        chosen = candidates.sort_values(
            "AtRisk_Probability",
            ascending=True,
        ).iloc[0]

    selected_cases.append(
        chosen.to_dict()
    )

selected_cases_df = pd.DataFrame(
    selected_cases
)

selected_cases_df



## Step 5 — Define graph-conditional SHAP prediction functions

For a given target student, SHAP perturbs only that student's encoded feature
vector.

The following remain fixed:

- graph edges;
- all neighbour features;
- all other student features;
- trained model parameters.

The returned value is the target student's predicted probability of being at risk.


In [ ]:

base_graph_x = (
    graph.x
    .detach()
    .clone()
)

def make_node_prediction_function(
    target_node_index
):
    def predict(candidate_vectors):
        candidate_vectors = np.asarray(
            candidate_vectors,
            dtype=np.float32,
        )

        outputs = []

        model.eval()

        for candidate in candidate_vectors:
            x_modified = (
                base_graph_x
                .detach()
                .clone()
            )

            x_modified[
                target_node_index
            ] = torch.tensor(
                candidate,
                dtype=torch.float32,
                device=device,
            )

            with torch.no_grad():
                logits = model(
                    x_modified,
                    graph.edge_index,
                )

                probability = torch.softmax(
                    logits,
                    dim=1,
                )[
                    target_node_index,
                    1,
                ]

            outputs.append(
                float(
                    probability
                    .cpu()
                    .item()
                )
            )

        return np.array(outputs)

    return predict



## Step 6 — Compute SHAP values

To keep the model-agnostic SHAP computation practical:

- a small training background sample is used;
- a reproducible sample of test students is used for global importance;
- all selected local cases are always included.

The encoded SHAP values are later aggregated back to the original questionnaire
feature names.


In [ ]:

rng = np.random.default_rng(
    REPRESENTATIVE_SEED
)

background_size = min(
    20,
    len(train_idx),
)

background_indices = rng.choice(
    train_idx,
    size=background_size,
    replace=False,
)

background_encoded = (
    X_all_encoded[
        background_indices
    ]
)

global_sample_size = min(
    12,
    len(test_idx),
)

global_indices = rng.choice(
    test_idx,
    size=global_sample_size,
    replace=False,
).tolist()

for case in selected_cases:
    index = int(
        case["Node_Index"]
    )

    if index not in global_indices:
        global_indices.append(index)

print(
    "Background students:",
    len(background_indices),
)
print(
    "Explained test students:",
    len(global_indices),
)


In [ ]:

all_shap_rows = []

for count, node_index in enumerate(
    global_indices,
    start=1,
):
    print(
        f"Explaining {count}/{len(global_indices)} "
        f"(node index {node_index})"
    )

    prediction_function = (
        make_node_prediction_function(
            int(node_index)
        )
    )

    explainer = shap.KernelExplainer(
        prediction_function,
        background_encoded,
    )

    shap_values = explainer.shap_values(
        X_all_encoded[
            int(node_index)
        ].reshape(1, -1),
        nsamples=120,
        silent=True,
    )

    values = np.asarray(
        shap_values
    ).reshape(-1)

    for feature_name, shap_value in zip(
        encoded_feature_names,
        values,
    ):
        all_shap_rows.append({
            "Node_Index": int(node_index),
            "Encoded_Feature": feature_name,
            "SHAP_Value": float(
                shap_value
            ),
        })

encoded_shap_df = pd.DataFrame(
    all_shap_rows
)

print("Encoded SHAP rows:", len(encoded_shap_df))



## Step 7 — Aggregate one-hot encoded features back to original variables

Categorical variables may become several one-hot encoded columns.

For thesis reporting, those encoded columns are grouped back under the original
questionnaire variable.


In [ ]:

def original_feature_from_encoded(
    encoded_name
):
    # ColumnTransformer prefixes:
    # numeric__Q10_MathsScore
    # categorical__Q6_Residence_Boarding
    stripped = encoded_name.split(
        "__",
        1,
    )[-1]

    # Exact numeric match first.
    if stripped in feature_columns:
        return stripped

    # For one-hot features, choose the longest
    # original column that is a prefix.
    matches = [
        column
        for column in feature_columns
        if stripped.startswith(
            column + "_"
        )
    ]

    if matches:
        return max(
            matches,
            key=len,
        )

    return stripped

encoded_shap_df[
    "Original_Feature"
] = encoded_shap_df[
    "Encoded_Feature"
].apply(
    original_feature_from_encoded
)

aggregated_shap_df = (
    encoded_shap_df
    .groupby(
        [
            "Node_Index",
            "Original_Feature",
        ],
        as_index=False,
    )["SHAP_Value"]
    .sum()
)

global_importance_df = (
    aggregated_shap_df
    .assign(
        Abs_SHAP=lambda frame:
        frame["SHAP_Value"].abs()
    )
    .groupby(
        "Original_Feature",
        as_index=False,
    )["Abs_SHAP"]
    .mean()
    .rename(
        columns={
            "Abs_SHAP":
            "Mean_Absolute_SHAP"
        }
    )
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    global_importance_df.head(15)
)


In [ ]:

# Global feature-importance figure.
top_global = global_importance_df.head(
    15
).sort_values(
    "Mean_Absolute_SHAP",
    ascending=True,
)

plt.figure(figsize=(9, 7))
plt.barh(
    top_global["Original_Feature"],
    top_global["Mean_Absolute_SHAP"],
)
plt.xlabel(
    "Mean absolute SHAP value"
)
plt.ylabel(
    "Original feature"
)
plt.title(
    "FairWarn-SHS global feature attribution"
)
plt.tight_layout()
plt.savefig(
    "figure_shap_global_importance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## Step 8 — Generate separate local explanation figures

In [ ]:

local_output_rows = []

for case in selected_cases:
    node_index = int(
        case["Node_Index"]
    )

    case_type_name = str(
        case["Case_Type"]
    )

    local = (
        aggregated_shap_df[
            aggregated_shap_df[
                "Node_Index"
            ].eq(node_index)
        ]
        .copy()
    )

    local["Abs_SHAP"] = (
        local["SHAP_Value"].abs()
    )

    local = local.sort_values(
        "Abs_SHAP",
        ascending=False,
    ).head(10)

    for _, row in local.iterrows():
        local_output_rows.append({
            "Node_Index": node_index,
            "Case_Type": case_type_name,
            "True_Label": int(
                case["True_Label"]
            ),
            "Predicted_Label": int(
                case["Predicted_Label"]
            ),
            "AtRisk_Probability": float(
                case["AtRisk_Probability"]
            ),
            "Original_Feature": row[
                "Original_Feature"
            ],
            "SHAP_Value": float(
                row["SHAP_Value"]
            ),
        })

    plot_local = local.sort_values(
        "SHAP_Value",
        ascending=True,
    )

    plt.figure(figsize=(9, 6))
    plt.barh(
        plot_local["Original_Feature"],
        plot_local["SHAP_Value"],
    )
    plt.axvline(
        0,
        linewidth=1,
    )
    plt.xlabel(
        "SHAP contribution to at-risk probability"
    )
    plt.ylabel(
        "Original feature"
    )
    plt.title(
        f"{case_type_name}: local FairWarn-SHS explanation"
    )
    plt.tight_layout()

    safe_name = (
        case_type_name
        .lower()
        .replace(" ", "_")
    )

    plt.savefig(
        f"figure_shap_{safe_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

local_cases_df = pd.DataFrame(
    local_output_rows
)

display(local_cases_df.head(20))



## Interpretation rule for the thesis

For this implementation:

- a **positive SHAP value** pushes the target student's prediction toward
  **at risk**;
- a **negative SHAP value** pushes the prediction toward **not at risk**;
- magnitude indicates relative contribution within the explanation.

Because the graph neighbourhood is held fixed, these values should be described as
**node-feature attributions conditional on the observed graph**, not as causal effects.


In [ ]:

metadata = {
    "model": "FairWarn-SHS",
    "fairness_lambda": FAIRNESS_LAMBDA,
    "representative_seed": REPRESENTATIVE_SEED,
    "seed_selection_rule": (
        "FairWarn-SHS seed with AUC-PR closest "
        "to the five-seed mean AUC-PR from Notebook 11."
    ),
    "explainer": "SHAP KernelExplainer",
    "explanation_scope": (
        "Target student's own encoded features, "
        "conditional on fixed graph structure and "
        "fixed features of all other nodes."
    ),
    "background_students": int(
        len(background_indices)
    ),
    "global_explained_test_students": int(
        len(global_indices)
    ),
    "kernel_nsamples": 120,
    "causal_claim": False,
    "edge_explanation_claim": False,
}

with open(
    OUTPUT_DIR
    / "fairwarn12_explainability_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

global_importance_df.to_csv(
    OUTPUT_DIR
    / "fairwarn12_shap_global_importance.csv",
    index=False,
)

local_cases_df.to_csv(
    OUTPUT_DIR
    / "fairwarn12_shap_local_cases.csv",
    index=False,
)

encoded_shap_df.to_csv(
    OUTPUT_DIR
    / "fairwarn12_encoded_shap_values.csv",
    index=False,
)

selected_cases_df.to_csv(
    OUTPUT_DIR
    / "fairwarn12_selected_case_types.csv",
    index=False,
)

from google.colab import files

download_list = [
    OUTPUT_DIR
    / "fairwarn12_shap_global_importance.csv",
    OUTPUT_DIR
    / "fairwarn12_shap_local_cases.csv",
    OUTPUT_DIR
    / "fairwarn12_explainability_metadata.json",
    "figure_shap_global_importance.png",
]

for filename in download_list:
    files.download(
        str(filename)
    )

for case in selected_cases:
    safe_name = (
        str(case["Case_Type"])
        .lower()
        .replace(" ", "_")
    )

    filename = (
        f"figure_shap_{safe_name}.png"
    )

    if Path(filename).exists():
        files.download(filename)
